## 22   PREPARING DATA

In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from matplotlib.ticker import MultipleLocator


In [3]:
pd.set_option("display.max_columns", None)

In [4]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent.parent
    print(script_dir)
    data_folder = script_dir /'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [5]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 257 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\UserPrevious experiments-preprocessed.json
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 229277 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\Data:Previous experiments-preprocessed.json


In [6]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [7]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

[6, 49, 257, 258]
Index([  0,   1,   2,   3,   4,   5,   7,   8,   9,  10,
       ...
       249, 250, 251, 252, 253, 254, 255, 256, 257, 258],
      dtype='int64', length=257)


In [8]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [9]:
timeSeriesData_BIG_Original = timeSeriesData_BIGraw.copy()
userInputData_Original = userInputDataRaw.copy()

In [10]:
userInputData_Original

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,time taken before insertion,time taken before insertion (seconds),time taken after insertion (seconds),time taken total (seconds),time taken total (seconds)-capped,time taken before insertion (seconds)-capped,time taken after insertion (seconds)-capped,Id=1:BME680:breathVocEquivalent MAX value VOC rolling average,Id=1:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent index of MAX value VOC rolling average,sensor with MAX value experiment,sensor with second MAX value experiment,x axis,y axis,x-y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,position of Id=0:BME680:breathVocEquivalent x-y axis,position of Id=1:BME680:breathVocEquivalent x-y axis,position of Id=2:BME680:breathVocEquivalent x-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
0,InsertingSourcePollutant,on,None,None,1.0,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.0,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01 00:00:00,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543,0 days 00:05:00,0 days 00:42:23,2543,300,2843,270,-30,299,4.3501,299.0,8.1928,243,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.25,3.0,"[-2.25, 3.0]",NaN,NaN,2.4,3.05,1.70,3.05,"[None, None]","[2.4, 3.05]","[1.7000000000000002, 3.05]",NaN,4.65,3.95
1,InsertingSourcePollutant,on,None,None,1.0,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.0,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02 00:00:00,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000,0 days 00:05:20,0 days 00:16:40,1000,320,1320,270,-30,299,1.1333,201.0,0.3304,80,NaN,NaN,Id=1:BME680:breathVocEquivalent MAX value VOC ...,Id=2:BME680:breathVocEquivalent MAX value VOC ...,-2.25,3.0,"[-2.25, 3.0]",NaN,NaN,2.4,3.05,1.70,3.05,"[None, None]","[2.4, 3.05]","[1.7000000000000002, 3.05]",NaN,4.65,3.95
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.0,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03 00:00:00,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248,0 days 00:10:58,0 days 00:04:08,248,658,906,270,-30,299,1.3923,235.0,2.2994,299,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.25,3.6,"[-2.25, 3.6]",NaN,NaN,2.4,3.05,1.70,3.05,"[None, None]","[2.4, 3.05]","[1.7000000000000002, 3.05]",NaN,4.68,3.99
3,InsertingSourcePollutant,on,None,None,1.0,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03 00:00:00,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88,0 days 00:06:02,0 days 00:01:28,88,362,450,270

In [11]:
userInputData_Original["room"].unique()

array(['Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55',
       'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90',
       'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0',
       'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'],
      dtype=object)

In [12]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space d
room_mask = userInputData_Original["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0','Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'])
mask = room_mask  
userInputData_Original = userInputData_Original.loc[mask]
timeSeriesData_BIG_Original = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.index)]

In [13]:
userInputData_Original

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,time taken before insertion,time taken before insertion (seconds),time taken after insertion (seconds),time taken total (seconds),time taken total (seconds)-capped,time taken before insertion (seconds)-capped,time taken after insertion (seconds)-capped,Id=1:BME680:breathVocEquivalent MAX value VOC rolling average,Id=1:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent index of MAX value VOC rolling average,sensor with MAX value experiment,sensor with second MAX value experiment,x axis,y axis,x-y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,position of Id=0:BME680:breathVocEquivalent x-y axis,position of Id=1:BME680:breathVocEquivalent x-y axis,position of Id=2:BME680:breathVocEquivalent x-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
7,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:04:09,2025-07-19 14:59:47,2025-07-19 15:08:10,2025-07-19 00:00:00,2025-07-19 14:59:49,2025-07-19 15:08:09,0 days 00:08:20,0 days 00:04:21,261,0 days 00:03:59,0 days 00:04:21,261,239,500,210,-30,239,0.2081,-30.0,0.5084,172,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"[-2.15, 2.2]",NaN,NaN,2.05,0.6,2.15,3.2,"[None, None]","[2.05, 0.6000000000000001]","[2.15, 3.2]",NaN,4.49,4.41
8,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:17:10,2025-07-19 15:14:13,2025-07-19 15:21:23,2025-07-19 00:00:00,2025-07-19 15:14:15,2025-07-19 15:21:21,0 days 00:07:06,0 days 00:02:56,176,0 days 00:04:10,0 days 00:02:56,176,250,426,221,-30,250,0.3916,250.0,1.0680,250,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"[-2.15, 2.2]",NaN,NaN,2.05,0.6,2.15,3.2,"[None, None]","[2.05, 0.6000000000000001]","[2.15, 3.2]",NaN,4.49,4.41
9,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:30:49,2025-07-19 15:25:07,2025-07-19 15:44:28,2025-07-19 00:00:00,2025-07-19 15:25:09,2025-07-19 15:44:27,0 days 00:19:18,0 days 00:05:41,341,0 days 00:13:37,0 days 00:05:41,341,817,1158,270,-30,299,1.4493,299.0,1.0056,299,NaN,NaN,Id=1:BME680:breathVocEquivalent MAX value VOC ...,Id=2:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"[-2.15, 2.2]",NaN,NaN,2.05,0.6,2.15,3.2,"[None, None]","[2.05, 0.6000000000000001]","[2.15, 3.2]",NaN,4.49,4.41
10,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 16:12:54,2025-07-19 16:04:22,2025-07-19 16:21:55,2025-07-19 00:00:00,2025-07-19 16:04:24,2025-07-19 16:21:54,0 days 00:17:30,0 days 00:08:31,511,0 days 00:08:59,0 days 00:08:31,511,539,1050,270,-30,299,1.7484,299.0,2.963

In [14]:
timeSeriesData_BIG_Original

,keys,sensors,VOC,after_insertion,original_value,datetime_timestamp,timestamp,seconds,seconds passed from insertionSource,VOC original,VOC rolling average,original seconds,VOC-capped,VOC rolling average-capped,VOC gradient,VOC rolling average gradient
seconds,,,,,,,,,,,,,,,,
-30,7,Id=2:BME680:breathVocEquivalent,0.119,False,False,2025-07-19 15:03:40,0 days 00:03:51,-30,-30,0.798525,0.1213,231,0.119,0.1213,-0.00060,-0.00060
-30,7,Id=1:BME680:breathVocEquivalent,0.217,False,True,2025-07-19 15:03:40,0 days 00:03:51,-30,-30,1.404333,0.2081,231,0.217,0.2081,-0.00070,-0.00070
-29,7,Id=2:BME680:breathVocEquivalent,0.121,False,False,2025-07-19 15:03:41,0 days 00:03:52,-29,-29,0.800500,0.1207,232,0.121,0.1207,-0.00055,-0.00055
-29,7,Id=1:BME680:breathVocEquivalent,0.216,False,False,2025-07-19 15:03:41,0 days 00:03:52,-29,-29,1.403337,0.2074,232,0.216,0.2074,-0.00060,-0.00060
-28,7,Id=2:BME680:breathVocEquivalent,0.124,False,True,2025-07-19 15:03:42,0 days 00:03:53,-28,-28,0.803212,0.1202,233,0.124,0.1202,-0.00045,-0.00045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,258,Id=1:BME680:breathVocEquivalent,1.421,True,True,2025-10-24 21:59:04,0 days 00:06:46,298,298,2.948842,1.4032,406,1.421,1.4032,-0.00500,-0.00500
298,258,Id=2:BME680:breathVocEquivalent,3.343,True,False,2025-10-24 21:59:04,0 days 00:06:46,298,298,5.567955,3.3352,406,3.343,3.3352,-0.00835,-0.00835
299,258,Id=0:BME680:breathVocEquivalent,2.143,True,False,2025-10-24 21:59:05,0 days 00:06:47,299,299,3.110278,2.1139,407,2.143,2.1139,0.01260,0.01260


In [15]:
column_to_transform = "x-y axis"


userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(lambda lst: [round(x,2) for x in lst if x is not None])

In [16]:
column_to_transform = "x-y axis"
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [17]:
userInputData_Original.loc[:,column_to_transform]

7      (-2.15, 2.2)
8      (-2.15, 2.2)
9      (-2.15, 2.2)
10     (-2.15, 2.2)
11     (-2.15, 2.2)
           ...     
254     (-0.5, 3.5)
255    (-2.95, 3.5)
256    (-2.95, 2.5)
257    (-2.95, 1.5)
258    (-2.95, 0.5)
Name: x-y axis, Length: 224, dtype: object

In [18]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                    for id in range(3)]

userInputData_Original.loc[:,sensors_position_columns] = userInputData_Original.loc[:,sensors_position_columns].applymap(lambda lst: [round(x,2) for x in lst if x is not None])
userInputData_Original.loc[:,sensors_euclidian_distance_columns] = userInputData_Original.loc[:,sensors_euclidian_distance_columns].round(2)


In [19]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
column_to_transform = sensors_position_columns
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [20]:
userInputData_Original

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,time taken before insertion,time taken before insertion (seconds),time taken after insertion (seconds),time taken total (seconds),time taken total (seconds)-capped,time taken before insertion (seconds)-capped,time taken after insertion (seconds)-capped,Id=1:BME680:breathVocEquivalent MAX value VOC rolling average,Id=1:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent index of MAX value VOC rolling average,sensor with MAX value experiment,sensor with second MAX value experiment,x axis,y axis,x-y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,position of Id=0:BME680:breathVocEquivalent x-y axis,position of Id=1:BME680:breathVocEquivalent x-y axis,position of Id=2:BME680:breathVocEquivalent x-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
7,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:04:09,2025-07-19 14:59:47,2025-07-19 15:08:10,2025-07-19 00:00:00,2025-07-19 14:59:49,2025-07-19 15:08:09,0 days 00:08:20,0 days 00:04:21,261,0 days 00:03:59,0 days 00:04:21,261,239,500,210,-30,239,0.2081,-30.0,0.5084,172,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"(-2.15, 2.2)",NaN,NaN,2.05,0.6,2.15,3.2,[],"[2.05, 0.6]","[2.15, 3.2]",NaN,4.49,4.41
8,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:17:10,2025-07-19 15:14:13,2025-07-19 15:21:23,2025-07-19 00:00:00,2025-07-19 15:14:15,2025-07-19 15:21:21,0 days 00:07:06,0 days 00:02:56,176,0 days 00:04:10,0 days 00:02:56,176,250,426,221,-30,250,0.3916,250.0,1.0680,250,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"(-2.15, 2.2)",NaN,NaN,2.05,0.6,2.15,3.2,[],"[2.05, 0.6]","[2.15, 3.2]",NaN,4.49,4.41
9,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:30:49,2025-07-19 15:25:07,2025-07-19 15:44:28,2025-07-19 00:00:00,2025-07-19 15:25:09,2025-07-19 15:44:27,0 days 00:19:18,0 days 00:05:41,341,0 days 00:13:37,0 days 00:05:41,341,817,1158,270,-30,299,1.4493,299.0,1.0056,299,NaN,NaN,Id=1:BME680:breathVocEquivalent MAX value VOC ...,Id=2:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"(-2.15, 2.2)",NaN,NaN,2.05,0.6,2.15,3.2,[],"[2.05, 0.6]","[2.15, 3.2]",NaN,4.49,4.41
10,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 16:12:54,2025-07-19 16:04:22,2025-07-19 16:21:55,2025-07-19 00:00:00,2025-07-19 16:04:24,2025-07-19 16:21:54,0 days 00:17:30,0 days 00:08:31,511,0 days 00:08:59,0 days 00:08:31,511,539,1050,270,-30,299,1.7484,299.0,2.9637,299,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breat

## MAKE THE DATA WITH TWO SENSORS AND DIFFERENT POSITIONS COMPATABLE

### MAKE THE POSITIONS OF THE X AXIS FOR EACH SENSOR NEGATIVE AT THE NEW DATA AND SEE WHAT SENSORS OF THE NEW SENSORS ARE CLOSER TO THE MODEL GIVEN ONES

In [21]:
mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
userInputData_Original.loc[mask]

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,time taken before insertion,time taken before insertion (seconds),time taken after insertion (seconds),time taken total (seconds),time taken total (seconds)-capped,time taken before insertion (seconds)-capped,time taken after insertion (seconds)-capped,Id=1:BME680:breathVocEquivalent MAX value VOC rolling average,Id=1:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent MAX value VOC rolling average,Id=2:BME680:breathVocEquivalent index of MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent MAX value VOC rolling average,Id=0:BME680:breathVocEquivalent index of MAX value VOC rolling average,sensor with MAX value experiment,sensor with second MAX value experiment,x axis,y axis,x-y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis,position of Id=0:BME680:breathVocEquivalent x-y axis,position of Id=1:BME680:breathVocEquivalent x-y axis,position of Id=2:BME680:breathVocEquivalent x-y axis,Euclidian distance to Id=0:BME680:breathVocEquivalent,Euclidian distance to Id=1:BME680:breathVocEquivalent,Euclidian distance to Id=2:BME680:breathVocEquivalent
7,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:04:09,2025-07-19 14:59:47,2025-07-19 15:08:10,2025-07-19 00:00:00,2025-07-19 14:59:49,2025-07-19 15:08:09,0 days 00:08:20,0 days 00:04:21,261,0 days 00:03:59,0 days 00:04:21,261,239,500,210,-30,239,0.2081,-30.0,0.5084,172,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"(-2.15, 2.2)",NaN,NaN,2.05,0.6,2.15,3.2,[],"[2.05, 0.6]","[2.15, 3.2]",NaN,4.49,4.41
8,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:17:10,2025-07-19 15:14:13,2025-07-19 15:21:23,2025-07-19 00:00:00,2025-07-19 15:14:15,2025-07-19 15:21:21,0 days 00:07:06,0 days 00:02:56,176,0 days 00:04:10,0 days 00:02:56,176,250,426,221,-30,250,0.3916,250.0,1.0680,250,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"(-2.15, 2.2)",NaN,NaN,2.05,0.6,2.15,3.2,[],"[2.05, 0.6]","[2.15, 3.2]",NaN,4.49,4.41
9,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 15:30:49,2025-07-19 15:25:07,2025-07-19 15:44:28,2025-07-19 00:00:00,2025-07-19 15:25:09,2025-07-19 15:44:27,0 days 00:19:18,0 days 00:05:41,341,0 days 00:13:37,0 days 00:05:41,341,817,1158,270,-30,299,1.4493,299.0,1.0056,299,NaN,NaN,Id=1:BME680:breathVocEquivalent MAX value VOC ...,Id=2:BME680:breathVocEquivalent MAX value VOC ...,-2.15,2.2,"(-2.15, 2.2)",NaN,NaN,2.05,0.6,2.15,3.2,[],"[2.05, 0.6]","[2.15, 3.2]",NaN,4.49,4.41
10,InsertingSourcePollutant,on,on,None,1.8,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1,NaN,NaN,1.1,None,None,2025-07-19 16:12:54,2025-07-19 16:04:22,2025-07-19 16:21:55,2025-07-19 00:00:00,2025-07-19 16:04:24,2025-07-19 16:21:54,0 days 00:17:30,0 days 00:08:31,511,0 days 00:08:59,0 days 00:08:31,511,539,1050,270,-30,299,1.7484,299.0,2.9637,299,NaN,NaN,Id=2:BME680:breathVocEquivalent MAX value VOC ...,Id=1:BME680:breat

In [22]:
def make_first_negative(v):
    if isinstance(v, list) and len(v) >= 2:
        return [-abs(v[0]), v[1]]
    return v


columns_to_grab= ["position of Id="+str(i)+":BME680:breathVocEquivalent x-y axis" for i in range(3)]
mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
userInputData_Original.loc[mask, columns_to_grab] = (
    userInputData_Original.loc[mask, columns_to_grab]
    .applymap(make_first_negative)
)
userInputData_Original.loc[mask,columns_to_grab]

,position of Id=0:BME680:breathVocEquivalent x-y axis,position of Id=1:BME680:breathVocEquivalent x-y axis,position of Id=2:BME680:breathVocEquivalent x-y axis
7,[],"[-2.05, 0.6]","[-2.15, 3.2]"
8,[],"[-2.05, 0.6]","[-2.15, 3.2]"
9,[],"[-2.05, 0.6]","[-2.15, 3.2]"
10,[],"[-2.05, 0.6]","[-2.15, 3.2]"
11,[],"[-2.05, 0.6]","[-2.15, 3.2]"
12,[],"[-2.05, 0.6]","[-2.15, 3.2]"
13,[],"[-2.05, 0.6]","[-2.15, 3.2]"
14,[],"[-2.05, 0.6]","[-2.15, 3.2]"
15,[],"[-2.05, 0.6]","[-2.15, 3.2]"
16,[],"[-2.05, 0.6]","[-2.15, 3.2]"


In [23]:
columns_to_transform = ["position of Id="+str(i)+":BME680:breathVocEquivalent x-y axis" for i in range(3)]
for column_to_transform in columns_to_transform:
    userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [24]:
userInputData_Original.loc[:,columns_to_transform]

,position of Id=0:BME680:breathVocEquivalent x-y axis,position of Id=1:BME680:breathVocEquivalent x-y axis,position of Id=2:BME680:breathVocEquivalent x-y axis
7,(),"(-2.05, 0.6)","(-2.15, 3.2)"
8,(),"(-2.05, 0.6)","(-2.15, 3.2)"
9,(),"(-2.05, 0.6)","(-2.15, 3.2)"
10,(),"(-2.05, 0.6)","(-2.15, 3.2)"
11,(),"(-2.05, 0.6)","(-2.15, 3.2)"
...,...,...,...
254,"(-1.85, 3.5)","(-0.5, 1.7)","(-2.25, 0.4)"
255,"(-1.85, 3.5)","(-0.5, 1.7)","(-2.25, 0.4)"
256,"(-1.85, 3.5)","(-0.5, 1.7)","(-2.25, 0.4)"
257,"(-1.85, 3.5)","(-0.5, 1.7)","(-2.25, 0.4)"


In [25]:
columns_to_grab= ["position of Id="+str(i)+":BME680:breathVocEquivalent x-y axis" for i in range(3)]
mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
positions_of_model_sensors = userInputData_Original.loc[mask,columns_to_grab].iloc[0]
positions_of_model_sensors

position of Id=0:BME680:breathVocEquivalent x-y axis              ()
position of Id=1:BME680:breathVocEquivalent x-y axis    (-2.05, 0.6)
position of Id=2:BME680:breathVocEquivalent x-y axis    (-2.15, 3.2)
Name: 7, dtype: object

In [26]:
columns_to_grab= ["position of Id="+str(i)+":BME680:breathVocEquivalent x-y axis" for i in range(3)]
mask = userInputData_Original["room"] != 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
positions_of_model_sensors = userInputData_Original.loc[mask,columns_to_grab].iloc[0]
positions_of_model_sensors

position of Id=0:BME680:breathVocEquivalent x-y axis    (-1.85, 3.5)
position of Id=1:BME680:breathVocEquivalent x-y axis     (-0.5, 1.7)
position of Id=2:BME680:breathVocEquivalent x-y axis    (-2.25, 0.4)
Name: 79, dtype: object

In [27]:
type(positions_of_model_sensors['position of Id=0:BME680:breathVocEquivalent x-y axis'])

tuple

### CHANGE NEW SENSOR ID 1 TO ID 2 AND ID 2 TO ID 0,FOR USERINPUT AND TIMESERIES

In [28]:
#FOR SENSOR ID:2 GIVE IT THE POSITION OF SENSOR 0 IN MODEL SENSORS AND CHANGE ID 2 TO ID 0
mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
new_position = pd.Series(data=np.empty(mask.sum()))
new_position = new_position.apply(lambda x: positions_of_model_sensors['position of Id=0:BME680:breathVocEquivalent x-y axis'] )

userInputData_Original.loc[mask,'position of Id=0:BME680:breathVocEquivalent x-y axis'] = new_position
#ALSO change the name of sensors in the corresponding time series from id:1 to id:2
time_series_mask = (timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.loc[mask].index)) & (timeSeriesData_BIG_Original["sensors"]=='Id=2:BME680:breathVocEquivalent')
timeSeriesData_BIG_Original.loc[time_series_mask,"sensors"] = 'Id=0:BME680:breathVocEquivalent'


#FOR SENSOR ID:1 GIVE IT THE POSITION OF SENSOR 2 IN MODEL SENSORS AND CHANGE ID 1 TO ID 2
mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
new_position = pd.Series(data=np.empty(mask.sum()))
new_position = new_position.apply(lambda x: positions_of_model_sensors['position of Id=2:BME680:breathVocEquivalent x-y axis'] )

userInputData_Original.loc[mask,'position of Id=2:BME680:breathVocEquivalent x-y axis'] = new_position

#ALSO change the name of sensors in the corresponding time series from id:1 to id:2
time_series_mask = (timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.loc[mask].index)) & (timeSeriesData_BIG_Original["sensors"]=='Id=1:BME680:breathVocEquivalent')
timeSeriesData_BIG_Original.loc[time_series_mask,"sensors"] = 'Id=2:BME680:breathVocEquivalent'

#MAKE THE position of Id=1:BME680:breathVocEquivalent x-y axis EMPTY LIST
mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
new_position = pd.Series(data=np.empty(mask.sum()))
new_position = new_position.apply(lambda x: [] )

userInputData_Original.loc[mask,'position of Id=1:BME680:breathVocEquivalent x-y axis'] = new_position

timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.loc[mask].index),"sensors"].unique()

array(['Id=0:BME680:breathVocEquivalent',
       'Id=2:BME680:breathVocEquivalent'], dtype=object)

In [29]:
mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1' ) & userInputData_Original[]


SyntaxError: invalid syntax (842685536.py, line 1)

### RECALCULATE THE EUCLIDIAN DISTANCES 

In [ ]:
source_position_name = "x-y axis"
source_position_column = np.array(userInputData_Original[source_position_name])

mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
sensor_names = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.loc[mask].index),"sensors"].unique()
for sensor_name in sensor_names:

    sensor_position_name = "position of "+ sensor_name + " x-y axis"
    sensor_position_column = np.array(userInputData_Original[sensor_position_name])
    print(source_position_column)
    print(sensor_position_column)

    distances = np.linalg.norm(
        source_position_column - sensor_position_column,
        axis=1
    )
    var_name = "Euclidian distance to "+sensor_name
    userInputData_Original.loc[:,var_name] = distances
    #round the distance
    userInputData_Original.loc[:,var_name] = userInputData_Original.loc[:,var_name].round(2)
    

## END MODIFY

In [ ]:
userInputData_Original.columns

In [ ]:

euclidian_distance_columns = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                               for id in range(3) ]

mask = userInputData_Original["are-doors-opened"] == "on"
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] == on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")
mask = userInputData_Original["are-doors-opened"] != "on"    
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] != on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")

In [ ]:
userInputData_Original['x-y axis'].unique()

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]=="on"].shape

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]!="on"].shape

In [ ]:
max_before= -30
max_after = 299

column_to_check = "time taken before insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] >max_before
print(f"{column_to_check} < {max_before}:\n {userInputData_Original.loc[mask,column_to_check]}")

column_to_check = "time taken after insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] < max_after
print(f"{column_to_check} < {max_after}:\n {userInputData_Original.loc[mask,column_to_check]}")
print(f"userInputData_Original rows {userInputData_Original.shape[0]}")

In [ ]:
userInputData_Original.columns

# LIBRARIES AND FUNCTIONS DECLARACTIONS

## LIBRARIES

In [ ]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score

import math
from itertools import combinations
from sklearn.preprocessing import RobustScaler

## MODELS

In [ ]:

# 1. Linear Regression (no hyperparameters to tune, just for completeness)
lr = LinearRegression()
lr_params = {}  # No parameters for simple LinearRegression

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# 3. Lasso Regression
lasso = Lasso()
lasso_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
}

# 4. ElasticNet
elastic = ElasticNet()
elastic_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# 5. Support Vector Regression
svr = SVR()
svr_params = {
    'kernel':["rbf"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}


# 7. Random Forest Regression
rf = RandomForestRegressor()
rf_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 8. Gradient Boosting Regression
gbr = GradientBoostingRegressor()
gbr_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [ 5, 10, 20,None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

models = {
    'LinearRegression': (lr, lr_params),
    'Ridge': (ridge, ridge_params),
    'Lasso': (lasso, lasso_params),
    'ElasticNet': (elastic, elastic_params),
    'SVR': (svr, svr_params),
   # 'RandomForest': (rf, rf_params),
    #'GradientBoosting': (gbr, gbr_params)
}

from scipy.optimize import least_squares


## FUNCTIONS

#### get_16_train_positions

In [ ]:
def get_16_train_positions(userInputData_Original:pd.DataFrame):
    column_to_check = "x-y axis"
    mask = (userInputData_Original["are-doors-opened"] != "on") & (userInputData_Original["room"] =='Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' ) & (userInputData_Original[column_to_check] != (None,None))
    training_positions = userInputData_Original.loc[mask,column_to_check].unique()
    return training_positions


#### get_Data_To_Be_Used

In [ ]:
def get_Data_To_Be_Used(userInputData_Original:pd.DataFrame,timeSeriesData_BIG_Original:pd.DataFrame,
                        keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)->(pd.DataFrame,pd.DataFrame):

    userInputData = userInputData_Original.copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.copy()

    # clear the data that doesn't fit to the 330 columns we want
    max_before= -30
    max_after = 299
    
    mask_before = (userInputData["time taken before insertion (seconds)-capped"] == max_before)
    max_after = (userInputData["time taken after insertion (seconds)-capped"] == max_after)
    mask = mask_before & max_after
 #   print(f"data which doesn't fit our criteria of size:{userInputData.loc[~mask,:].index}")
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    if keepOpenDoorData:
        mask = (userInputData["are-doors-opened"] == "on") | (userInputData["are-doors-opened"] != "on")

    else:
        mask =  userInputData["are-doors-opened"] != "on"

    if anyOtherMaskToBeUsed is not None:
        mask = mask & anyOtherMaskToBeUsed
    userInputData = userInputData.loc[mask].copy()
    #grab all the data that are contained in those experiments
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
        
    
    if drop_other_positions & (not closeDoorTraining_openDoorTest):
        #from the experiments with door open, drop the positions which doesn't fit to the 16 source position we are going to check the model
        #also drop the experiments with the None values
        axis_list = get_16_train_positions(userInputData_Original)
        axis_mask = userInputData["x-y axis"].isin(axis_list)
        
        mask = axis_mask 
        userInputData = userInputData.loc[mask]
        #grab all the data that are contained in those experiments
        timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    #drop only none data   
    elif (not drop_other_positions) & (not closeDoorTraining_openDoorTest):
        mask = userInputData["x-y axis"] != (None,None)
        userInputData = userInputData.loc[mask]
        #grab all the data that are contained in those experiments
        timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    return userInputData,timeSeriesData_BIG
        

#### build_X_keys_And_Y_target_Arrays

In [ ]:
def build_X_keys_And_Y_target_Arrays(userInputData,closeDoorTraining_openDoorTest = False)->(np.array,np.array):
  
    if closeDoorTraining_openDoorTest :
         pass

    else:
        keys = np.array(userInputData.index)
        keys_numpy = keys.reshape(-1,1)
   

        columns_to_pass = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                                   for id in range(3) ]
        column_size = len(columns_to_pass)
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_euclidian_distance_numpy = target_variables.reshape(-1,column_size)
     
    
    
        columns_to_pass = ["x-y axis"]
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_positions_numpy = np.stack(target_variables[:,0])  
        
        X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions = train_test_split(
            keys_numpy,target_variables_euclidian_distance_numpy,target_variables_positions_numpy,
            test_size=0.16,
            stratify=target_variables_euclidian_distance_numpy,
            random_state=42
        )
    
    return X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions

#### build_X_from_dataframe

In [ ]:
def build_X_from_dataframe(keys_array, df, column_to_take):
    # flatten [[159],[196]...] → [159,196,...]
    keys = keys_array.ravel()

    # number of keys
    N = len(keys)

    # number of time steps per experiment (should be constant)
    example_key = keys[0]
    time_values = df[df["keys"] == example_key]
    T = len(time_values)

    # allocate X
    X = np.zeros((N, T))

    # fill X
    for i, key in enumerate(keys):
        rows = df[df["keys"] == key]
        X[i, :] = rows[column_to_take].values

    return X

#### get_Data_Per_Sensor

In [ ]:
def get_Data_Per_Sensor(userInputData,timeSeriesData_BIG) ->pd.DataFrame:
    df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    sensors_names = np.sort(timeSeriesData_BIG["sensors"].unique())
    dfs_by_sensor = {
        sensor: df_filtered.loc[df_filtered["sensors"]==sensor]
        for sensor in sensors_names
    }
    return dfs_by_sensor


In [ ]:
def build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take ="VOC rolling average")->(np.array,np.array):
    dfs_by_sensor = get_Data_Per_Sensor(userInputData,timeSeriesData_BIG)
    X_train = np.empty((len(dfs_by_sensor),X_train_keys.shape[0],330))
    
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_train[i,:,:]= build_X_from_dataframe(X_train_keys,data_per_sensor,column_to_take)
    
    X_test = np.empty((len(dfs_by_sensor),X_test_keys.shape[0],330))
        
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_test[i,:,:]= build_X_from_dataframe(X_test_keys,data_per_sensor,column_to_take)

    return X_train,X_test

#### fix_negatives_by_interpolation

In [ ]:

def fix_negatives_by_interpolation(x):
    x = x.copy()

    idx = np.arange(len(x))
    neg_idx = np.argwhere(x < 0).flatten()
    
    neg_idx_checked = {idx : False for idx in neg_idx}
    print(neg_idx_checked)
    for index,check in neg_idx_checked.items():
        #was already checked go to another loop
        if check is True:
            continue
        else:    
            neg_idx_checked[index] = True
            start = index - 1 
            end = index + 1 
            found_positive_value = False
            while found_positive_value is False:
                #check if key exists in neg_idx_checked
                if end in neg_idx_checked:
                    neg_idx_checked[end] = True
                    end = end + 1
                else:
                    found_positive_value = True
           
            first_negative_idx = start+1
            last_negative_idx = end 
           
            x[first_negative_idx:last_negative_idx] = np.interp(idx[first_negative_idx:last_negative_idx], [idx[start],idx[end]], [x[start],x[end]])
    
    return x

In [ ]:
def fix_negatives_by_interpolation_3_dimension_Array(X,X_name):
    set_of_samples_with_negative_values = set()
    neg_element_position = np.argwhere(X < 0)
    if neg_element_position.size > 0:
        print(f"{X_name} has negative values at indices:")
        print(neg_element_position)
        for position in neg_element_position:
            
            set_of_samples_with_negative_values.add((position[0],position[1]))   
            
        for sample_with_negative_values in set_of_samples_with_negative_values:
            
            X[*sample_with_negative_values,:] = fix_negatives_by_interpolation(X[*sample_with_negative_values,:])
    else:
        print(f"{X_name} has no negative values at indices")

    return X        

In [ ]:




def fix_negatives_by_interpolation_X_train_X_test(X_train,X_test):
    
    X_train = fix_negatives_by_interpolation_3_dimension_Array(X_train,"X_train")
    X_test  = fix_negatives_by_interpolation_3_dimension_Array(X_test,"X_test") 
    return X_train,X_test

#### TRIM FIRST COLUMNS

In [ ]:
def trimFirstColumns(X_train,X_test,columns_to_keep = 15):
    actual_columns_to_keep = abs(-30) + columns_to_keep -1


    X_train =  X_train[:,:,actual_columns_to_keep:]
    X_test  =  X_test[:,:,actual_columns_to_keep:] 
    return X_train,X_test    

#### scaleExponentialValues

In [ ]:

def scaleExponentialValues(X_train,Y_train_Euclidians):
        for i in range(X_train.shape[0]):
            y = np.asarray(Y_train_Euclidians[:,i])
            
            
            powers = 2 - 0.5 * ( (y - y.min()) / (y.max() - y.min()) )
            X_train[i,:,:] = X_train[i,:,:] ** powers[:, np.newaxis]
        return X_train
    
   

#### create_The_Arrays_For_The_Models

In [ ]:

def create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take ="VOC rolling average",keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None):
    
    userInputData,timeSeriesData_BIG = get_Data_To_Be_Used(userInputData_Original,timeSeriesData_BIG_Original,
                                                           keepOpenDoorData,drop_other_positions,closeDoorTraining_openDoorTest,anyOtherMaskToBeUsed)
    
    X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions = build_X_keys_And_Y_target_Arrays(userInputData)  
    
    X_train,X_test = build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take)
    X_train,X_test = fix_negatives_by_interpolation_X_train_X_test(X_train,X_test)
  #  X_train,X_test = trimFirstColumns(X_train,X_test,columns_to_keep = 60)
   # X_train = scaleExponentialValues(X_train,Y_train_Euclidians)
    if apply_functions_to_X_data is not None and apply_functions_to_X_data is not np.nan:
       for modify_function in apply_functions_to_X_data:
           if modify_function is not None and modify_function is not np.nan:
               X_train,X_test =  modify_function(X_train,X_test)
    return X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG

In [ ]:

    
def findPCAcomponentsCovered(X_train):     
    
    for X_train_per_sensor in X_train:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train_per_sensor)
        # Step 2: Apply PCA
        pca = PCA()
        X_pca = pca.fit_transform(X_scaled)
        
        # Step 3: Explained variance
        explained_variance = pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)
        
        # Only display first 10 components max
        max_components = min(10, len(explained_variance))
        ev_to_plot = explained_variance[:max_components]
        cum_to_plot = cumulative_variance[:max_components]
        
        # Step 4: Bar plot of cumulative explained variance
        plt.figure(figsize=(8,5))
        plt.bar(range(1, max_components + 1), cum_to_plot)
        plt.xlabel('Number of PCA components')
        plt.ylabel('Cumulative explained variance')
        plt.title('Cumulative explained variance (first 10 components)')
        plt.grid(True, axis='y')
        plt.show()
        
        # Step 5: Optimal number of components for ~90% variance
        optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
        print("Optimal number of components to explain ~90% variance:", optimal_components)


In [ ]:
def printPCA2components(X_train,Y_train_Euclidians):
    for i in range(X_train.shape[0]):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train[i])
        
        # --- Step 2: Reduce to 2 components ---
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        
        # --- Step 3: Scatter plot with hue based on Y_train ---
        plt.figure(figsize=(8,6))
        
        # Using seaborn for easy color scaling
        sns.scatterplot(
            x=X_pca[:,0], 
            y=X_pca[:,1], 
            hue=Y_train_Euclidians[:,i],              # color represents Y_train
             palette="coolwarm",         # continuous color palette
            s=80,                      # marker size
            alpha=0.8
        )
        
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.title("2D PCA of X_train colored by Y_train")
        plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.show()

#### target_to_weight

In [ ]:
def target_to_weight(y):
    y = np.asarray(y)
   
    y_min = y.min()
    y_max = y.max()
   
    if y_max == y_min:
        return np.zeros_like(y)  # or np.ones_like(y), depending on intent

    weights = 2 - 4 * (y - y_min) / (y_max - y_min)
    return weights

#### run_grid_search_per_model

In [ ]:

def run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params):
        PCA__n_components = [2,3,4,5,6,8,10]

        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']

        results = {}
   
        pipe = Pipeline([
                                    #    ('log scaler',FunctionTransformer(np.log1p, validate=True)),

               ('scaler', StandardScaler()),

           # ('scaler', RobustScaler()),
                 ('PCA', PCA()),
                 ('model', model)
             ])
        # Build a correct param grid:
        param_grid = {
            **{'model__' + k: v for k, v in params.items()}
        }
        if name not in estimators_with_no_scaling_need:
            param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='neg_mean_squared_error',
            verbose=2
        )
        
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor(X_train, y_train,cv_number,verbose,models):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
     #       print(best_result["name"])
            best_result["score"] = results["score"]
     #       print(best_result["score"])
            best_result["model"] = results["model"]
       #     print(best_result["model"])
            best_result["parameters"] = results["parameters"]
       #     print(best_result["parameters"])
    return best_result


def run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, y_train,cv_number,verbose,models):

    best_results_for_all_sensors = {}

    for i in range(X_train.shape[0]):
        best_results_for_all_sensors[i]  =run_grid_search_find_optimal_model_per_sensor(X_train[i,:,:], y_train[:,i],cv_number,verbose,models)

    return best_results_for_all_sensors

In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
def getPredictedValues(X_test,Y_test_Euclidians,best_results_for_all_sensors):

   predict_Euclidians = np.empty((Y_test_Euclidians.shape))
    
    
   for i in range(X_test.shape[0]):
        print(f"For sensor with id={i}")
        predict_Euclidians[:,i] = best_results_for_all_sensors[i]["model"].predict(X_test[i,:,:])
    
   return predict_Euclidians
       

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,models):
    cv_number = 5
    verbose = 0
    
    best_results_for_all_sensors = run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, Y_train_Euclidians,cv_number,verbose,models)
    
    Y_predict_Euclidians = getPredictedValues(X_test,Y_test_Euclidians,best_results_for_all_sensors)
    plantAllBaseSensorsData(Y_test_Euclidians, Y_predict_Euclidians,"Euclidian distance",userInputData)
    return Y_predict_Euclidians

In [ ]:
def grabPositionOfSensors(userInputData):
 # --- SENSOR POINTS (red true, blue/orange predicted) ---
    sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
    point_of_sensors = []
    mask = userInputData["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    for sensors_position_column in sensors_position_columns:
        
        point_of_sensors.append(userInputData.loc[mask,sensors_position_column].iloc[0]) 

    return   point_of_sensors 

#### plot_pred_vs_actual

In [ ]:
def plot_pred_vs_actual(y_test, y_pred, sensor_name, AXIS,userInputData):

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # convert to 2D format
    if y_test.ndim == 1:
        y_test = y_test.reshape(-1,1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1,1)

    # Euclidean error
    EucDis = np.linalg.norm(y_test - y_pred, axis=1).mean()
    
    print(f"\n--- SENSOR {sensor_name} ---")
    print(f"MSE on test set: {mse:.4f}")
    print(f"R^2 on test set: {r2:.4f}")
    print(f"Euclidean distance mean: {EucDis:.4f}")

    # If 2D case
    if y_test.shape[1] == 2:

        plt.figure(figsize=(8,7))

        # --- TRUE (blue) ---
        plt.scatter(y_test[:,0], y_test[:,1], label="True", alpha=0.7, color="blue")
        
        # Print ID next to each true point
        for idx, (x,y) in enumerate(y_test):
            plt.text(x, y, str(idx), color="blue", fontsize=9)

        # --- PREDICTED (orange) ---
        plt.scatter(y_pred[:,0], y_pred[:,1], label="Predicted", alpha=0.7, color="orange")
        
        # Print ID next to each predicted point
        for idx, (x,y) in enumerate(y_pred):
            plt.text(x, y, str(idx), color="orange", fontsize=9)

        point_of_sensors = grabPositionOfSensors(userInputData)
        if point_of_sensors is not None:
            point_of_sensors = np.array(point_of_sensors)

            # TRUE sensor points → BLUE  
            plt.scatter(point_of_sensors[:,0], point_of_sensors[:,1],
                        marker="X", s=200, color="red", label="Sensor Base Points")

        plt.legend()
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.title(f"Sensor {sensor_name} - True vs Predicted")
        plt.grid(True)
        plt.show()

    else:
        # fallback for 1D case
        plt.figure(figsize=(7,6))
        plt.scatter(y_test, y_pred, s=40)
        plt.plot([y_test.min(), y_test.max()], 
                 [y_test.min(), y_test.max()], linestyle='--')
        plt.xlabel("True")
        plt.ylabel("Predicted")
        plt.title(f"{sensor_name} (1D) predictions vs actual")
        plt.grid(True)
        plt.show()
    return -mse, -EucDis, r2   
    
def plantAllBaseSensorsData(y_test, y_pred,AXIS,userInputData):
    for i in range(y_test.shape[1]):
        AXIS = "Euclidian distances"
        plot_pred_vs_actual(y_test[:,i], y_pred[:,i],i,AXIS,userInputData)


In [ ]:
def trilateration_least_squares(points, radii, tol=5e-2):
    def fun(p):
        x, y = p
        return [np.hypot(x - px, y - py) - r for (px, py), r in zip(points, radii)]

    x0 = [np.mean([px for px,_ in points]),
          np.mean([py for _,py in points])]

    res = least_squares(fun, x0=x0)
    x, y = res.x

    # Residuals close to zero?
    boundary_ok = np.all(np.abs(fun([x, y])) < tol)

    # Check if point lies inside all circles
    inside_ok = all(
        np.hypot(x - px, y - py) <= r + tol
        for (px, py), r in zip(points, radii)
    )

    has_common_point = inside_ok  # this is the correct criterion

    return has_common_point, (x, y)
 

#### findIntersectionOfCircles

In [ ]:
#Take the three points of the sensors
def findIntersectionOfCircles(predict_Euclidians,Y_test_Positions,userInputData):
    
    point_of_sensors = grabPositionOfSensors(userInputData)

    predict_Positions = np.empty((predict_Euclidians.shape[0],2))
    threshold_values =  np.empty((predict_Euclidians.shape[0],1))
    for i in range(predict_Euclidians.shape[0]):
        threshold,point = trilateration_least_squares(point_of_sensors,predict_Euclidians[i,:])
        predict_Positions[i,:] = point
        threshold_values[i] = threshold
    
    MSE, EUCLIDIAN, R2 = plot_pred_vs_actual(Y_test_Positions,predict_Positions,"Final","Point predicted",userInputData)
    return predict_Positions,threshold_values,MSE, EUCLIDIAN, R2

#### PIPELINE

In [ ]:

def runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params ={},apply_functions_to_X_data = []):
    print(extra_params)
    (X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,
        timeSeriesData_BIG) = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data,column_to_take,**extra_params)
    print(Y_test_Euclidians.shape)
    printPCA2components(X_train,Y_train_Euclidians)
    predict_Euclidians = trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,models)

    predict_Positions,threshold_values,MSE, EUCLIDIAN, R2 = findIntersectionOfCircles(predict_Euclidians,Y_test_Positions,userInputData)

    return (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
    Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,predict_Euclidians,predict_Positions,threshold_values,
           MSE,EUCLIDIAN,R2)

In [ ]:
def circle_intersections(c1, r1, c2, r2):
    x0, y0 = c1
    x1, y1 = c2

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    # No intersection
    if d > r1 + r2 or d < abs(r1 - r2) or d == 0:
        return []

    # Compute a, h
    a = (r1**2 - r2**2 + d**2) / (2*d)
    h = math.sqrt(max(0, r1**2 - a**2))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    xs1 = xm + h * dy / d
    ys1 = ym - h * dx / d
    xs2 = xm - h * dy / d
    ys2 = ym + h * dx / d

    return [(xs1, ys1), (xs2, ys2)]

# ------------------------------------------------------------
# Collect intersection points that lie inside all circles
# ------------------------------------------------------------
def intersection_points_of_all_circles(points, radii, tol=1e-6):
    candidates = []

    # pairwise intersections
    for (p1, r1), (p2, r2) in combinations(zip(points, radii), 2):
        pts = circle_intersections(p1, r1, p2, r2)
        candidates.extend(pts)

    # keep only points that lie inside ALL circles
    inside = []
    for x, y in candidates:
        if all(math.hypot(x - px, y - py) <= r + tol
               for (px, py), r in zip(points, radii)):
            inside.append((x, y))

    return inside

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------
def plot_circles_and_intersections(points, radii,i,at):
    fig, ax = plt.subplots(figsize=(6, 6))

    # Draw all circles
    for (px, py), r in zip(points, radii):
        circle = plt.Circle((px, py), r, fill=False, lw=2)
        ax.add_patch(circle)
        ax.plot(px, py, 'ko')  # centers

    # Compute intersections
    ints = intersection_points_of_all_circles(points, radii)

    # Plot intersection points
    for x, y in ints:
        ax.plot(x, y, 'ro', markersize=8)

    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_title("Circles and Intersection Points for row " + str(i) + " at "+at)
    plt.show()

## FILTER FUNCTIONS

In [ ]:
def cutOutliers(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], q25, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], q25, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], q25, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], q25, q75)        

    return X_train,X_test       

In [ ]:
def cutOutliersUpper(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], None, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], None, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersUpperPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], None, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], None, q75)        

    return X_train,X_test       

In [ ]:
def createGradient(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,j,:] = np.gradient(X_train[i,j,:])
            
    for i in range(X_test.shape[0]):
        for j in range(X_test.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_test[i,j,:] = np.gradient(X_test[i,j,:])        

    return X_train,X_test       

In [ ]:
def rollingWindows(X_train:np.array,X_test:np.array,window_size=15)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,j,:] =   (pd.Series(X_train[i,j,:])
                                .rolling(window_size,center= True ,min_periods=1)
                                .mean()
                                .to_numpy()
                             )       
    for i in range(X_test.shape[0]):
        for j in range(X_test.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_test[i,j,:] =  (pd.Series(X_test[i,j,:])
                                .rolling(window_size,center= True ,min_periods=1)
                                .mean()
                                .to_numpy()
                             )       

    return X_train,X_test       

# FIND THE VARIABLES

## CREATE DATAFRAMES AND FUNCTION TO PRINT

In [ ]:
functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
functions_to_apply_Gradient = [None,createGradient]
functions_to_apply_Outliers_names = [X.__name__ if X is not None else X for X in functions_to_apply_Outliers]
functions_to_apply_Gradient_names = [X.__name__ if X is not None else X for X in functions_to_apply_Gradient]

column_to_take =["VOC original","VOC","VOC rolling average"]
room_cases = ["Only closed","All data"]
multi_index = pd.MultiIndex.from_product([room_cases,column_to_take,functions_to_apply_Outliers_names,functions_to_apply_Gradient_names],names= ["room cases" ,"column to take","Outlier functions","Gradient functions"])
for idx in multi_index:
    print(idx)

In [ ]:
columns = ["MSE","EUCLIDIAN","R2"]
parametersResultsGathered = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGathered

In [ ]:
def loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,applyWindow=False,applyWindowAfterOutlier=False):
    print(column_to_take)
    print(extra_params)
    functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
    functions_to_apply_Gradient = [None,createGradient]

    for function_Outlier in functions_to_apply_Outliers:
        outlier_name = function_Outlier.__name__ if function_Outlier is not None else np.nan
        print("outlier_name "+str(outlier_name))

        for function_Gradient in functions_to_apply_Gradient:

            gradient_name = function_Gradient.__name__ if function_Gradient is not None else np.nan
            print("gradient_name "+str(gradient_name))

            if applyWindow:
                apply_functions_to_X_data = [rollingWindows,function_Outlier,function_Gradient]
            elif applyWindowAfterOutlier: 
                apply_functions_to_X_data = [function_Outlier,rollingWindows,function_Gradient]

            else:
                apply_functions_to_X_data = [function_Outlier,function_Gradient]

            (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
                Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
                 predict_Euclidians,points,threshold_values,
                 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

            idx = (room_case, column_to_take,outlier_name ,gradient_name )
            parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]] = [MSE, EUCLIDIAN, R2]
            print(parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]])
    return parametersResultsGathered      
            # MSE,EUCLIDIAN,R2 the values i want to insert 

## RUN THE FUNCTIONS

### ONLY CLOSED

#### VOC Original

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
columns = ["MSE","EUCLIDIAN","R2"]
parametersResultsGatheredRolling = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGatheredRolling

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGatheredRolling = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,applyWindow=True,applyWindowAfterOutlier=False)

In [ ]:
parametersResultsGatheredRolling.loc[room_case]

In [ ]:
column_to_take = "VOC"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGatheredRolling = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,applyWindow=True,applyWindowAfterOutlier=False)

In [ ]:
parametersResultsGatheredRolling.loc[room_case]

In [ ]:
columns = ["MSE","EUCLIDIAN","R2"]
parametersResultsGatheredRollingAfter = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGatheredRollingAfter

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGatheredRollingAfter = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,applyWindow=False,applyWindowAfterOutlier=True)

In [ ]:
parametersResultsGatheredRollingAfter.loc[room_case]

#### VOC

In [ ]:
column_to_take = "VOC"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}
apply_functions_to_X_data = [function_Outlier,function_Gradient]
parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,,applyWindow=False,applyWindowAfterOutlier=True)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC rolling average

In [ ]:
column_to_take = "VOC rolling average"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
    key=lambda col: col.abs()
)
df_sorted

### ALL DATA

#### VOC rolling average

#### VOC Original

In [ ]:
column_to_take = "VOC original"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC

In [ ]:
column_to_take = "VOC"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
column_to_take = "VOC rolling average"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
parametersResultsGathered.head(60)

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
)
df_sorted

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

In [ ]:
column_to_take = "VOC original"
extra_params = { "keepOpenDoorData" :False}
apply_functions_to_X_data = [createGradient,cutOutliersUpperPerColumn]

(predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
 predict_Euclidians,points,threshold_values,
 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

In [ ]:
Y_test_Positions.shape

In [ ]:
column_to_take = "VOC"
extra_params = { "keepOpenDoorData" :True}
apply_functions_to_X_data = [createGradient,cutOutliersUpper]

(predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
 predict_Euclidians,points,threshold_values,
 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

In [ ]:
Y_test_Positions.shape

### END

## APPLY MACHINE LEARNING MODEL

In [ ]:
functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
functions_to_apply_Gradient = [None,createGradient]
functions_to_apply_Outliers_names = [X.__name__ if X is not None else X for X in functions_to_apply_Outliers]
functions_to_apply_Gradient_names = [X.__name__ if X is not None else X for X in functions_to_apply_Gradient]

column_to_take =["VOC original","VOC","VOC rolling average"]
room_cases = ["Only closed","All data"]
multi_index = pd.MultiIndex.from_product([room_cases,column_to_take,functions_to_apply_Gradient_names,functions_to_apply_Outliers_names],names= ["room cases" ,"column to take","Gradient functions","Outlier functions"])
for idx in multi_index:
    print(idx)

In [ ]:
columns = ["MSE","EUCLIDIAN","R2"]
parametersResultsGatheredMachineLearning = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGatheredMachineLearning

In [ ]:
def loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered):
    print(column_to_take)
    print(extra_params)
    functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
    functions_to_apply_Gradient = [None,createGradient]

    
    for function_Gradient in functions_to_apply_Gradient:

        for function_Outlier in functions_to_apply_Outliers:
            outlier_name = function_Outlier.__name__ if function_Outlier is not None else np.nan
            gradient_name = function_Gradient.__name__ if function_Gradient is not None else np.nan
            print("outlier_name "+str(outlier_name))
            print("gradient_name "+str(gradient_name))


            apply_functions_to_X_data = [function_Outlier,function_Gradient]

            (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
                Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
                 predict_Euclidians,points,threshold_values,
                 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

            idx = (room_case, column_to_take, gradient_name, outlier_name)
            parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]] = [MSE, EUCLIDIAN, R2]
            print(parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]])
    return parametersResultsGathered      
            # MSE,EUCLIDIAN,R2 the values i want to insert 

# FEATURE SEARCHING